In [1]:
def format_input(entry, prompt_style="alpaca"):
    if prompt_style == "alpaca":
        instruction_text = (
            f"Below is an instruction that describes a task. "
            f"Write a response that appropriately completes the request."
            f"\n\n### Instruction:\n{entry['instruction']}"
        )

        input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""

        return instruction_text + input_text
    elif prompt_style == "phi-3":
        punc = entry["instruction"][-1]
        user_request = f"<|user|>\n{entry['instruction'][:-1]}"
        if entry["input"]:
            user_request += f": {entry['input']}"
        else:
            user_request += punc
        return user_request
    else:
        raise ValueError(f"'{prompt_style}' is not a recognized prompt style.")

In [2]:
from dotenv import load_dotenv

load_dotenv(".env")
import os

from openai import OpenAI

client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.getenv("LLM_KEY"))


def query_model(prompt, model="anthropic/claude-haiku-4.5"):
    params = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "seed": 0,
        "temperature": 0.1,
        "max_tokens": 200,
    }

    response = client.chat.completions.create(**params)
    return response.choices[0].message.content

In [3]:
from tqdm import tqdm


def generate_model_scores(
    json_data, json_key, prompt_style="alpaca", model="anthropic/claude-haiku-4.5"
):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        prompt = (
            f"Given the input `{format_input(entry, prompt_style=prompt_style)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        score = query_model(prompt, model)
        try:
            scores.append(int(score))
        except ValueError:
            print(f"Could not convert score: {score}")
            continue

    return scores

In [4]:
import json
from tqdm import tqdm

### Base Model Score (Alpaca Style Prompts, Instruction Loss Not Ignored)

In [7]:
file_path = "instruction-data-with-response-base.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [02:00<00:00,  1.09s/it]

Number of scores: 110 of 110
Average score: 34.05



### Alpaca Style Prompts, Instruction Loss Ignored

In [8]:
file_path = "instruction-data-with-response-instruction-ignore.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [01:59<00:00,  1.09s/it]

Number of scores: 110 of 110
Average score: 38.99



### Phi-3 Style Prompts, Instruction Loss Not Ignored

(Note that we do not need to change the prompt-style for formatting, as we are not giving the inputs to the trained models but rather only interested in the contents. It is even better to keep the styling the same for a fairer judgement)

In [9]:
file_path = "instruction-data-with-response-phi-3.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [02:09<00:00,  1.18s/it]

Number of scores: 110 of 110
Average score: 33.95



### Phi-3 Style Prompts, Instruction Loss Ignored

In [11]:
file_path = "instruction-data-with-response-phi-3-instruction-ignore.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [01:56<00:00,  1.05s/it]

Number of scores: 110 of 110
Average score: 34.97



### Alpaca Style Prompts, Instruction Loss Ignored, LoRA

In [5]:
file_path = "instruction-data-with-response-lora.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [02:06<00:00,  1.15s/it]

Number of scores: 110 of 110
Average score: 24.11



### Alpaca Style Prompts, Instruction Loss Ignored, Fine-Tuned on Alpaca Dataset

In [5]:
file_path = "instruction-data-with-response-alpaca-dataset-finetune.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [01:54<00:00,  1.04s/it]

Number of scores: 110 of 110
Average score: 40.39



### Alpaca Style Prompts, Instruction Loss Ignored, Fine-Tuned on Alpaca Dataset, LoRA

In [6]:
file_path = "instruction-data-with-response-alpaca-dataset-finetune-lora.json"
with open(file_path, "r") as file:
    test_data = json.load(file)

scores = generate_model_scores(test_data, "model_response")
print(f"Number of scores: {len(scores)} of {len(test_data)}")
print(f"Average score: {sum(scores) / len(scores):.2f}\n")

Scoring entries: 100%|██████████| 110/110 [01:41<00:00,  1.09it/s]

Number of scores: 110 of 110
Average score: 30.66

